# Week 63, Day 1: Latent variable models and the evidence lower bound

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
from scipy.stats import multivariate_normal as mvn
g0 = np.random.default_rng(631)
d, k = 12, 2
W = g0.normal(size=(d, k)); mu = g0.normal(size=d); s2 = 0.5
Z = g0.normal(size=(500, k))
X = Z @ W.T + mu + np.sqrt(s2) * g0.normal(size=(500, d))
M = W.T @ W + s2 * np.eye(k)

def log_evidence(x):
    return mvn.logpdf(x, mu, W @ W.T + s2 * np.eye(d))

def posterior(x):
    return np.linalg.solve(M, W.T @ (x - mu)), s2 * np.linalg.inv(M)
print("ready: a 12-dimensional probabilistic PCA model with 2 latents, and 500 observations")

### Exercise: The ELBO in closed form

Write elbo(x, m, S) for a Gaussian q = N(m, S) in the reconstruction form: the expected log-likelihood under q, E_q[log N(x; W z + mu, s2 I)], minus KL(q || N(0, I)). Use the closed forms: the expected squared error is |x - mu - W m|^2 + trace(W S W^T).

In [ ]:
def kl_gauss(m1, S1, m2, S2):
    """KL( N(m1, S1) || N(m2, S2) )"""
    k_ = len(m1); S2i = np.linalg.inv(S2)
    return 0.5 * (np.trace(S2i @ S1) + (m2 - m1) @ S2i @ (m2 - m1) - k_ + np.linalg.slogdet(S2)[1] - np.linalg.slogdet(S1)[1])

def elbo(x, m, S):
    sq = np.sum((x - mu - W @ m) ** 2) + np.trace(W @ S @ W.T)
    recon = ...
    return recon - kl_gauss(m, S, np.zeros(k), np.eye(k))

m0, S0 = posterior(X[0])
print(f"log evidence {log_evidence(X[0]):.4f}")
print(f"ELBO at the exact posterior {elbo(X[0], m0, S0):.4f}")
print(f"ELBO with the mean shifted by 0.5 and the covariance doubled {elbo(X[0], m0 + 0.5, 2 * S0):.4f}")

In [ ]:
import numpy as np
def test_exact():
    "with q equal to the exact posterior, the ELBO equals the log evidence"
    m, S = posterior(X[0])
    assert abs(elbo(X[0], m, S) - log_evidence(X[0])) < 1e-8
def test_gap():
    "with a worse q, the gap equals the KL divergence from q to the posterior"
    m, S = posterior(X[0]); m2, S2 = m + 0.5, 2 * S
    assert abs((log_evidence(X[0]) - elbo(X[0], m2, S2)) - kl_gauss(m2, S2, m, S)) < 1e-8

test_exact()
test_gap()
print("All checks passed")

### Exercise: A Monte Carlo ELBO

Estimate the same ELBO by sampling: draw z from q, average log p(x | z) + log p(z) - log q(z). This is how the ELBO is computed when no closed form exists.

In [ ]:
gm = np.random.default_rng(6311)
mq, Sq = m0 + 0.5, 2 * S0
zs = gm.multivariate_normal(mq, Sq, size=20000)
log_lik = mvn.logpdf(X[0] - zs @ W.T - mu, np.zeros(d), s2 * np.eye(d))
log_prior = mvn.logpdf(zs, np.zeros(k), np.eye(k))
log_q = mvn.logpdf(zs, mq, Sq)
elbo_mc = ...
print(f"Monte Carlo ELBO {elbo_mc:.4f} (closed form {elbo(X[0], mq, Sq):.4f}); standard error {np.std(log_lik + log_prior - log_q) / np.sqrt(20000):.4f}")

In [ ]:
import numpy as np
def test_mc():
    "the Monte Carlo ELBO with 20,000 samples is within 0.05 of the closed form"
    assert abs(elbo_mc - elbo(X[0], m0 + 0.5, 2 * S0)) < 0.05

test_mc()
print("All checks passed")

### Exercise: Learning an encoder from samples

Simulate 50,000 (z, x) pairs from the model, regress z on x - mu by least squares to get a linear encoder, and compare it with the exact posterior-mean matrix M^-1 W^T.

In [ ]:
ge = np.random.default_rng(6312)
Zs = ge.normal(size=(50000, k))
Xs = Zs @ W.T + mu + np.sqrt(s2) * ge.normal(size=(50000, d))
E_fit = ...                                            # k x d: z is approximately E_fit @ (x - mu)
E_exact = np.linalg.solve(M, W.T)
print("largest difference between fitted and exact encoder:", np.abs(E_fit - E_exact).max().round(4))

In [ ]:
import numpy as np
def test_encoder():
    "a linear map fitted by regressing sampled latents on observations matches the exact posterior-mean matrix"
    assert np.allclose(E_fit, np.linalg.solve(M, W.T), atol=0.05)

test_encoder()
print("All checks passed")